# 12.2 - Designing Reliability

**Module 12 - Production Deployment** | Netsetos GenAI Engineering

This notebook works through Designing Reliability hands-on: The reliability math: why every dependency erodes uptime; Timeouts and deadlines: bound every wait; Retry with exponential backoff and full jitter; The circuit breaker: stop hammering a dead service; Fallback and graceful degradation: something beats nothing; Bulkheads and idempotent retries: contain and deduplicate.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

A single reproducibility note. Every block in this notebook is deterministic - the "random" jitter picks and timings are scripted, not sampled - so runs are repeatable and no seed library is even needed.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A comment-only cell, not executable logic. It flags that the notebook deliberately avoids live randomness and real network calls so every output is reproducible and keyless.

**How the code works - step by step**
- **The comment** - signals that values throughout the notebook are seeded/scripted rather than sampled, so your output matches the printed `# Output:` blocks exactly.

**In one line:** everything here is deterministic on purpose.

**What you'll see:** (no output - environment prep)

## 1 - The reliability math: why every dependency erodes uptime

Before any pattern, the number that motivates all of them. Availability is measured in **nines**, and how you wire services decides which way those nines move: dependencies **in series** multiply your uptime *down*, while a **parallel** fallback multiplies it *up*. This is the whole reason fallback is not optional.

In [ ]:
# RELIABILITY MATH - dependencies multiply your uptime DOWN, redundancy multiplies it UP.
# Availability in "nines" (99.9% = three-nines). This is the whole reason fallback exists. No key.

def nines(a): return f"{a*100:.4f}% ({str(a*100).count('9')}-nines approx)"

# SERIES: a chain where EVERY dependency must work -> availabilities MULTIPLY.
deps = {"LLM API": 0.999, "vector DB": 0.999, "auth": 0.999}
series = 1.0
for a in deps.values(): series *= a
print("A chain of dependencies (all must work) multiplies DOWN:")
for name, a in deps.items(): print(f"  {name:10} {a*100:.1f}%")
print(f"  => composite {series*100:.2f}%  (WORSE than any single one: more deps = less uptime)")

# PARALLEL: a fallback where EITHER can serve -> only both-down takes you down.
primary, fallback = 0.99, 0.99
parallel = 1 - (1 - primary) * (1 - fallback)
print(f"\nA 99% primary ALONE:                     {primary*100:.2f}%")
print(f"A 99% primary + a 99% fallback (parallel): {parallel*100:.2f}%  (a second 9 for free)")

# ERROR BUDGET: the downtime an SLO allows.
HOURS_YEAR = 365.25 * 24
for a in (0.999, 0.9999):
    down_h = (1 - a) * HOURS_YEAR
    print(f"\nSLO {a*100:.2f}% -> error budget {down_h:.2f} h/year ({down_h*60/12:.0f} min/month)")
print("\nTakeaway: every dependency erodes uptime; a fallback (parallel) is the only thing that buys it back.")
# Output:
# A chain of dependencies (all must work) multiplies DOWN:
#   LLM API    99.9%
#   vector DB  99.9%
#   auth       99.9%
#   => composite 99.70%  (WORSE than any single one: more deps = less uptime)
#
# A 99% primary ALONE:                     99.00%
# A 99% primary + a 99% fallback (parallel): 99.99%  (a second 9 for free)
#
# SLO 99.90% -> error budget 8.77 h/year (44 min/month)
#
# SLO 99.99% -> error budget 0.88 h/year (4 min/month)
#
# Takeaway: every dependency erodes uptime; a fallback (parallel) is the only thing that buys it back.

**Explanation**

A pure arithmetic demo - no model call, just the availability algebra that every later pattern serves. It computes a series composite (all dependencies must work), a parallel composite (either can serve), and the downtime an SLO permits, to show that redundancy is the only thing that raises uptime above your weakest dependency.

**How the code works - step by step**
- **`nines(a)`** - a tiny formatter that prints an availability as a percentage.
- **SERIES block** - multiplies three 99.9% dependencies together (`0.999**3`), showing the composite drops *below* any single one.
- **PARALLEL block** - computes `1 - (1-primary)*(1-fallback)` for two 99% providers, so you are down only if *both* fail.
- **ERROR BUDGET block** - for a 99.9% and a 99.99% SLO, multiplies `(1-a)` by hours-per-year to get the allowed downtime.

**In one line:** series multiplies uptime down, parallel multiplies it back up, and the SLO fixes your downtime budget.

**What you'll see:** The series chain composites to 99.70% (worse than any 99.9% link); a lone 99% primary plus a 99% fallback jumps to 99.99%; and the error budget prints as 8.77 h/year for three-nines versus 0.88 h/year for four-nines.

## 2 - Timeouts and deadlines: bound every wait

The simplest reliability bug everyone ships first is a call with **no timeout** - it can hang for minutes while holding a worker. The fix is two layers: a **per-call timeout** that caps one call, and a **deadline budget** that caps the whole multi-hop request by being decremented at each hop, so a slow early dependency cannot starve the later ones.

In [ ]:
# TIMEOUTS & DEADLINES - never wait forever. A per-call timeout caps one call; a DEADLINE BUDGET
# caps the whole request across hops, so a slow early dependency cannot starve the later ones.
# A tick clock (ms) stands in for real time; deterministic, no key.

def call_with_timeout(name, work_ms, timeout_ms):
    if work_ms > timeout_ms:
        return None, f"TIMEOUT after {timeout_ms}ms (the call needed {work_ms}ms)"
    return f"{name} ok", f"done in {work_ms}ms"

# A single slow call against a 2000ms timeout:
_, msg = call_with_timeout("slow-llm", work_ms=5000, timeout_ms=2000)
print("Single call with a timeout:")
print(f"  {msg}  -> the worker is freed instead of hanging forever")

# A DEADLINE BUDGET of 900ms shared across a 3-hop chain:
print("\nDeadline budget 900ms across a 3-hop chain:")
budget = 900
for hop, cost in [("retrieve", 300), ("rerank", 400), ("generate", 500)]:
    if cost > budget:
        print(f"  {hop:9} needs {cost}ms but only {budget}ms left -> ABORT (fail fast, do not overrun)")
        break
    budget -= cost
    print(f"  {hop:9} took {cost}ms, {budget}ms left")
print("\nTakeaway: bound every wait; a deadline budget stops one slow hop from blowing the whole request.")
# Output:
# Single call with a timeout:
#   TIMEOUT after 2000ms (the call needed 5000ms)  -> the worker is freed instead of hanging forever
#
# Deadline budget 900ms across a 3-hop chain:
#   retrieve  took 300ms, 600ms left
#   rerank    took 400ms, 200ms left
#   generate  needs 500ms but only 200ms left -> ABORT (fail fast, do not overrun)
#
# Takeaway: bound every wait; a deadline budget stops one slow hop from blowing the whole request.

**Explanation**

A deterministic clock demo where a tick count in milliseconds stands in for real time - no real waiting or network. It shows a single call being aborted by its timeout, then walks a shared 900ms budget down a three-hop chain until a hop that would overrun is cut off.

**How the code works - step by step**
- **`call_with_timeout`** - returns a TIMEOUT result when the work exceeds the timeout, otherwise an ok result; models freeing the worker on overrun.
- **Single-call demo** - a 5000ms call against a 2000ms timeout is aborted rather than allowed to hang.
- **Deadline-budget loop** - starts at 900ms and subtracts each hop's cost (`retrieve` 300, `rerank` 400, `generate` 500), aborting the hop that needs more than remains.

**In one line:** cap one call with a timeout, cap the whole request with a budget that shrinks per hop.

**What you'll see:** The single call prints a TIMEOUT after 2000ms; the chain logs retrieve (600ms left), rerank (200ms left), then generate needing 500ms with only 200ms left so it ABORTs - fail fast instead of overrun.

## 3 - Retry with exponential backoff and full jitter

Many failures are **transient** - a rate limit, a brief 5xx, a network blip - and retrying a moment later just works. But naive retries make things worse: retry the *wrong* errors and you burn attempts, retry *without jitter* and every client retries at the same instant (a thundering herd). The fix is transient-only retry with exponential backoff plus full jitter, capped at about four attempts.

In [ ]:
# RETRY with EXPONENTIAL BACKOFF + FULL JITTER - recover transient failures without a thundering herd.
# Retry ONLY transient errors; back off exponentially; add full jitter so clients do not synchronize.
# Jitter is random(0, window); we script the fractions here so the demo is deterministic. No key.

TRANSIENT = {429, 500, 502, 503, 504}          # rate limit + server errors + timeouts -> retry
def should_retry(status): return status in TRANSIENT   # 401/403/400/context-overflow -> do NOT retry

BASE, CAP, MAX_ATTEMPTS = 1.0, 8.0, 4          # 4 total attempts is the LLM sweet spot
jitter_frac = [0.5, 0.3, 0.8]                   # illustrative "random(0,1)" picks (deterministic here)
# The call returns 503 twice, then succeeds on the 3rd attempt.
statuses = [503, 503, 200]
print("Retry a transient failure with full jitter (sleep = random(0, min(cap, base*2^n))):")
for attempt in range(MAX_ATTEMPTS):
    status = statuses[attempt] if attempt < len(statuses) else 200
    if status == 200:
        print(f"  attempt {attempt+1}: 200 OK -> success"); break
    if not should_retry(status):
        print(f"  attempt {attempt+1}: {status} non-transient -> STOP (retrying would waste time)"); break
    window = min(CAP, BASE * 2**attempt)
    sleep = round(jitter_frac[attempt] * window, 2)   # full jitter picks inside [0, window]
    print(f"  attempt {attempt+1}: {status} transient -> backoff window {window:.0f}s, jittered sleep {sleep}s")

# A non-transient error is not retried:
print("\nA 401 (auth) is NOT retried:")
print(f"  should_retry(401) = {should_retry(401)}  -> fail fast, do not burn 3 more attempts")

# A 429 can carry a Retry-After header - the SERVER tells you exactly how long to wait.
print("\nRetry-After overrides your computed backoff:")
computed = round(jitter_frac[0] * min(CAP, BASE * 2**0), 2)
retry_after = 3.0                                    # seconds, read from the 429 response header
print(f"  your computed jittered backoff: {computed}s")
print(f"  the 429 Retry-After header:      {retry_after}s -> obey the server, wait {max(retry_after, computed)}s")

# Thundering herd: without jitter, all clients retry at the SAME instant.
print("\nThundering herd (5 clients fail together, backoff window 4s):")
print("  NO jitter -> all 5 retry at exactly t=4s (a synchronized spike re-crushes the service)")
print("  FULL jitter -> the 5 spread across [0s, 4s] (a steady trickle the service can absorb)")
# Output:
# Retry a transient failure with full jitter (sleep = random(0, min(cap, base*2^n))):
#   attempt 1: 503 transient -> backoff window 1s, jittered sleep 0.5s
#   attempt 2: 503 transient -> backoff window 2s, jittered sleep 0.6s
#   attempt 3: 200 OK -> success
#
# A 401 (auth) is NOT retried:
#   should_retry(401) = False  -> fail fast, do not burn 3 more attempts
#
# Retry-After overrides your computed backoff:
#   your computed jittered backoff: 0.5s
#   the 429 Retry-After header:      3.0s -> obey the server, wait 3.0s
#
# Thundering herd (5 clients fail together, backoff window 4s):
#   NO jitter -> all 5 retry at exactly t=4s (a synchronized spike re-crushes the service)
#   FULL jitter -> the 5 spread across [0s, 4s] (a steady trickle the service can absorb)

**Explanation**

A scripted retry driver: the jitter fractions and the sequence of statuses are hard-coded so the demo is deterministic instead of actually sleeping or sampling. It shows a transient failure recovering, a non-transient error stopping immediately, a Retry-After header overriding your computed backoff, and the herd contrast.

**How the code works - step by step**
- **`TRANSIENT` set + `should_retry`** - only 429/500/502/503/504 are retryable; 401/403/400/context-overflow are not.
- **Retry loop** - for each attempt computes the backoff window `min(cap, base*2**attempt)` and a jittered sleep inside it; two 503s retry, then a 200 succeeds.
- **401 demo** - shows `should_retry(401)` is False, so the client fails fast.
- **Retry-After block** - takes `max(retry_after, computed)`, obeying the server's header over your own backoff.
- **Thundering-herd note** - contrasts 5 clients retrying at the same t=4s (no jitter) versus spread across [0s,4s] (full jitter).

**In one line:** retry only transient errors, back off exponentially, jitter to avoid the herd, and honor Retry-After.

**What you'll see:** Attempts 1-2 show 503 with backoff windows 1s/2s and jittered sleeps 0.5s/0.6s, attempt 3 succeeds; `should_retry(401)` prints False; Retry-After 3.0s wins over the 0.5s computed backoff; and the herd lines contrast a synchronized t=4s spike with a smooth [0s,4s] spread.

## 4 - The circuit breaker: stop hammering a dead service

Retries handle a *brief* outage; when a provider is *properly* down, retrying every request just piles load on a service that cannot answer. The **circuit breaker** is the escalation - a per-dependency state machine that trips **open** after enough failures and fast-fails without touching the dead service, then probes for recovery.

In [ ]:
# THE CIRCUIT BREAKER - stop hammering a dead service. A per-dependency state machine:
# CLOSED (calls flow, count failures) -> OPEN (fast-fail after fail_max) -> HALF-OPEN (probe after
# a cooldown) -> CLOSED on success. A tick timeline drives it deterministically, no key.

class CircuitBreaker:
    def __init__(self, fail_max=3, cooldown=2):
        self.fail_max, self.cooldown = fail_max, cooldown
        self.state, self.fails, self.opened_at = "CLOSED", 0, None
    def call(self, tick, provider_up):
        if self.state == "OPEN":
            if tick - self.opened_at >= self.cooldown:
                self.state = "HALF-OPEN"                 # cooldown elapsed -> probe
            else:
                return "fast-fail (open, not calling)"
        # CLOSED or HALF-OPEN: actually attempt the call
        if provider_up:
            self.state, self.fails = "CLOSED", 0          # success closes it
            return "ok"
        self.fails += 1
        if self.fails >= self.fail_max or self.state == "HALF-OPEN":
            self.state, self.opened_at = "OPEN", tick      # trip open
            return "FAIL -> tripped OPEN"
        return "FAIL (still closed)"

cb = CircuitBreaker(fail_max=3, cooldown=2)
downtime = {0: False, 1: False, 2: False, 3: False, 4: True, 5: True}  # recovers at tick 4
print("Provider is DOWN ticks 0-3, recovers at tick 4. fail_max=3, cooldown=2:")
for tick in range(6):
    result = cb.call(tick, downtime[tick])
    print(f"  tick {tick}: state={cb.state:9} -> {result}")
print("\nTakeaway: after 3 failures the breaker OPENS and fast-fails; a cooldown probe closes it on recovery.")
# Output:
# Provider is DOWN ticks 0-3, recovers at tick 4. fail_max=3, cooldown=2:
#   tick 0: state=CLOSED    -> FAIL (still closed)
#   tick 1: state=CLOSED    -> FAIL (still closed)
#   tick 2: state=OPEN      -> FAIL -> tripped OPEN
#   tick 3: state=OPEN      -> fast-fail (open, not calling)
#   tick 4: state=CLOSED    -> ok
#   tick 5: state=CLOSED    -> ok
#
# Takeaway: after 3 failures the breaker OPENS and fast-fails; a cooldown probe closes it on recovery.

**Explanation**

A small state-machine class driven over a deterministic tick timeline where a dict says whether the provider is up each tick. It encodes the CLOSED -> OPEN -> HALF-OPEN -> CLOSED lifecycle so you can watch the breaker trip, fast-fail, probe, and recover without any real service.

**How the code works - step by step**
- **`__init__`** - stores `fail_max`, `cooldown`, and starts CLOSED with a zero fail count.
- **`call`** - when OPEN, fast-fails until the cooldown elapses, then flips to HALF-OPEN to probe; on a live call, a success closes and resets, a failure increments and trips OPEN once `fail_max` is hit (or on any half-open miss).
- **Driver loop** - runs ticks 0-5 with the provider down 0-3 and recovering at tick 4, printing state and result each tick.

**In one line:** count failures, trip open, fast-fail through the cooldown, then probe and close on recovery.

**What you'll see:** Ticks 0-1 FAIL while still closed, tick 2 trips OPEN, tick 3 fast-fails without calling, and ticks 4-5 print ok as the half-open probe finds the recovered provider and closes the breaker.

## 5 - Fallback and graceful degradation: something beats nothing

Now compose failover with a floor. **Fallback** tries providers in priority order with health tracking and the first success wins; but when *every* provider is down, the right answer is not a 500 - it is **graceful degradation** down a ladder (cache, then cheaper, then a canned reply) so the user always gets something.

In [ ]:
# FALLBACK & GRACEFUL DEGRADATION - something beats nothing. Try providers in priority order with
# health tracking; when ALL providers fail, DEGRADE down a ladder (cache -> cheaper -> canned) so the
# user always gets SOMETHING instead of an error page. Scripted health, deterministic, no key.

def serve(providers, cache_hit):
    for name, up in providers:                      # 1) health-tracked fallback chain
        if up:
            return f"answer from {name}", name
    # 2) all providers down -> degrade down the ladder
    if cache_hit:
        return "stale cached answer (a few minutes old)", "cache"
    return "canned: 'We are experiencing high demand - here is our FAQ link.'", "canned"

print("Scenario A - primary+secondary down, tertiary up:")
r, via = serve([("primary", False), ("secondary", False), ("tertiary", True)], cache_hit=False)
print(f"  served by {via}: {r}")

print("\nScenario B - ALL providers down, cache has a stale copy:")
r, via = serve([("primary", False), ("secondary", False), ("tertiary", False)], cache_hit=True)
print(f"  degraded to {via}: {r}")

print("\nScenario C - ALL providers down, cache empty:")
r, via = serve([("primary", False), ("secondary", False), ("tertiary", False)], cache_hit=False)
print(f"  degraded to {via}: {r}")

print("\nTakeaway: a health-tracked chain, then a degradation ladder - the user ALWAYS gets a response.")
# Output:
# Scenario A - primary+secondary down, tertiary up:
#   served by tertiary: answer from tertiary
#
# Scenario B - ALL providers down, cache has a stale copy:
#   degraded to cache: stale cached answer (a few minutes old)
#
# Scenario C - ALL providers down, cache empty:
#   degraded to canned: canned: 'We are experiencing high demand - here is our FAQ link.'
#
# Takeaway: a health-tracked chain, then a degradation ladder - the user ALWAYS gets a response.

**Explanation**

A single scripted `serve` function with hard-coded provider health and a cache-hit flag - no real calls. It demonstrates the fallback-then-degrade contract across three scenarios, proving the function never returns an error: it always returns some response.

**How the code works - step by step**
- **`serve`** - loops the provider list and returns the first healthy one; if all are down, it degrades to a cached answer when `cache_hit`, otherwise to a canned FAQ response.
- **Scenario A** - primary and secondary down, tertiary up: the chain falls through to tertiary.
- **Scenario B** - all providers down but cache has a stale copy: degrade to cache.
- **Scenario C** - all providers down and cache empty: degrade to a canned response.

**In one line:** try providers by health, and when all fail, step down the ladder so a response always comes back.

**What you'll see:** Scenario A is served by tertiary, B degrades to a stale cached answer, and C degrades to the canned 'high demand - here is our FAQ link' response - every scenario returns something, never an error.

## 6 - Bulkheads and idempotent retries: contain and deduplicate

Two patterns that make the rest safe. A **bulkhead** gives each dependency its own bounded concurrency pool, so one slow dependency floods only *its* compartment and cannot consume every worker. And because you now retry, any retried **write** must carry an **idempotency key** or it will execute twice - retries and idempotency are a package deal.

In [ ]:
# BULKHEADS & IDEMPOTENT RETRIES - contain failures, and make retries safe. A BULKHEAD bounds the
# concurrency PER dependency (like a ship's watertight compartments) so one slow/failing dependency
# cannot consume every worker. And a retried write must be IDEMPOTENT or it runs twice. No key.

# --- Bulkhead: a slow dependency floods, but only its own 2-slot pool ---
def admit(calls, pool_size):
    running, rejected = [], []
    for c in calls:
        (running if len(running) < pool_size else rejected).append(c)
    return running, rejected

running, rejected = admit([f"slow-{i}" for i in range(6)], pool_size=2)
print("Bulkhead - 6 calls hit a SLOW dependency with a 2-slot pool:")
print(f"  running (in its pool): {running}")
print(f"  rejected/shed:         {rejected}  <- the fast dependency's pool is UNTOUCHED (isolated)")

# --- Idempotency: a retried write executes once with a key, twice without ---
charges, seen = [], set()
def charge_naive(order):      charges.append(order)                       # no key -> double-charges
def charge_keyed(order, key):
    if key in seen: return "duplicate ignored"
    seen.add(key); charges.append(order); return "charged"

charge_naive("A"); charge_naive("A")                                     # a retry delivers it twice
naive_count = charges.count("A")
charge_keyed("B", "idem-1"); charge_keyed("B", "idem-1")                 # same key twice
print("\nIdempotency - the SAME write delivered twice (a retry):")
print(f"  naive (no key): order A charged {naive_count} times  <- BUG: the retry double-charged")
print(f"  keyed:          order B charged {charges.count('B')} time   <- the key made the retry a no-op")
print("\nTakeaway: bulkheads isolate a failure to one compartment; idempotency keys make retries safe.")
# Output:
# Bulkhead - 6 calls hit a SLOW dependency with a 2-slot pool:
#   running (in its pool): ['slow-0', 'slow-1']
#   rejected/shed:         ['slow-2', 'slow-3', 'slow-4', 'slow-5']  <- the fast dependency's pool is UNTOUCHED (isolated)
#
# Idempotency - the SAME write delivered twice (a retry):
#   naive (no key): order A charged 2 times  <- BUG: the retry double-charged
#   keyed:          order B charged 1 time   <- the key made the retry a no-op
#
# Takeaway: bulkheads isolate a failure to one compartment; idempotency keys make retries safe.

**Explanation**

Two independent mini-demos in one cell, both deterministic. The first is an admission-control function that caps how many calls a pool accepts; the second contrasts a naive write that double-charges on retry with a keyed write that dedupes.

**How the code works - step by step**
- **`admit`** - fills a pool up to `pool_size` and sheds the rest; run with 6 calls and a 2-slot pool to show isolation.
- **`charge_naive`** - appends every call, so a retried order A lands twice (the bug).
- **`charge_keyed`** - checks an idempotency key against a `seen` set and ignores duplicates, so a twice-delivered order B charges once.

**In one line:** bulkheads shed a flood into one compartment; idempotency keys turn a retried write into a no-op.

**What you'll see:** The bulkhead runs slow-0 and slow-1 and sheds slow-2 through slow-5; the idempotency demo prints order A charged 2 times (naive bug) versus order B charged 1 time (the key made the retry a no-op).

## 7 - The resilient client: all patterns, one wrapper

In production these patterns wrap each other, and the **order matters**. This block composes timeout, circuit breaker, retry, fallback, and degradation into one client and traces a single request through every layer - the key insight being that an open breaker fast-fails *before* any retry is wasted on a known-dead provider.

In [ ]:
# THE RESILIENT CLIENT - all patterns, one wrapper, and the ORDER matters. A call is bounded by a
# TIMEOUT, gated by a per-provider CIRCUIT BREAKER, re-attempted by RETRY, moved on by FALLBACK, and
# floored by DEGRADATION. We trace one request through every layer. Scripted, deterministic, no key.

def resilient_call(providers, cache_hit):
    trace = []
    for name, breaker_open, attempts in providers:     # FALLBACK: try each provider in order
        if breaker_open:                                # CIRCUIT BREAKER: skip a provider that is open
            trace.append(f"{name}: breaker OPEN -> fast-fail, skip (no retries wasted)")
            continue
        for i, ok in enumerate(attempts):               # RETRY (each attempt is TIMEOUT-bounded)
            if ok:
                trace.append(f"{name}: attempt {i+1} within timeout -> OK")
                return "answer from " + name, trace
            trace.append(f"{name}: attempt {i+1} transient fail -> retry with backoff")
    # DEGRADATION: everything failed -> the floor
    trace.append("all providers exhausted -> degrade to " + ("cache" if cache_hit else "canned response"))
    return ("stale cache" if cache_hit else "canned response"), trace

# provider A breaker is OPEN; provider B fails once then succeeds on retry.
result, trace = resilient_call(
    providers=[("A", True, []), ("B", False, [False, True])],
    cache_hit=False)
print("One request through the resilient client (order: timeout -> breaker -> retry -> fallback -> degrade):")
for line in trace: print(f"  {line}")
print(f"  RESULT: {result}")
print("\nTakeaway: composed layers turn a fragile call into one that survives an open breaker AND a transient fail.")
# Output:
# One request through the resilient client (order: timeout -> breaker -> retry -> fallback -> degrade):
#   A: breaker OPEN -> fast-fail, skip (no retries wasted)
#   B: attempt 1 transient fail -> retry with backoff
#   B: attempt 2 within timeout -> OK
#   RESULT: answer from B
#
# Takeaway: composed layers turn a fragile call into one that survives an open breaker AND a transient fail.

**Explanation**

A single scripted `resilient_call` that threads one request through all five layers, with each provider's breaker state and per-attempt outcomes hard-coded so the trace is deterministic. It is the capstone that shows the layers reinforcing rather than fighting each other.

**How the code works - step by step**
- **Outer provider loop** - the FALLBACK layer, trying each provider in priority order.
- **`breaker_open` check** - the CIRCUIT BREAKER layer, skipping an open provider with no retries wasted.
- **Inner attempts loop** - the RETRY layer (each attempt notionally TIMEOUT-bounded), retrying transient failures until one succeeds.
- **Final fallthrough** - the DEGRADATION floor, returning cache or a canned response when everything is exhausted.
- **Driver** - provider A's breaker is open and provider B fails once then succeeds on retry.

**In one line:** breaker before retry before fallback before degrade - the ordering that never burns a retry on a dead provider.

**What you'll see:** The trace prints A skipped (breaker OPEN, no retries wasted), B attempt 1 transient-failing into a backoff retry, B attempt 2 succeeding within timeout, and RESULT: answer from B.

## 8 - Production libraries: install the real thing

You hand-rolled every pattern to see it clearly; in production you compose libraries instead. This cell prepares the environment for the real tenacity / pybreaker / pyresilience stack described alongside it.

> The pip line is commented out - uncomment it on Colab or a fresh environment.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


**Explanation**

A commented-out dependency install, not runnable logic. It marks the transition from hand-rolled demos to the battle-tested libraries (tenacity for retry, pybreaker for the breaker, pyresilience to unify them) that implement Steps 3-7 in a few decorators.

**How the code works - step by step**
- **The `!pip install` comment** - installs `python-dotenv` (and, when you extend it, the resilience libraries) once uncommented.

**In one line:** uncomment to install the production resilience stack.

**What you'll see:** (no output - the install line is commented out)

## 9 - Load provider keys from the environment

The multi-provider fallback chain is best exercised with real keys, so this cell wires up the standard provider environment variables. Any one key is enough to start; two or more light up the failover demos.

> **Needs provider API keys** to run against real endpoints (set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, and/or `GOOGLE_API_KEY`) - the notebook's own demos are keyless.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Environment prep, not a model call. It uses `os.environ.setdefault` to declare the three provider key slots without overwriting anything you already have set, and deliberately reads keys from the environment rather than hardcoding them.

**How the code works - step by step**
- **`import os`** - to touch the process environment.
- **Three `os.environ.setdefault(...)`** - declare `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, and `GOOGLE_API_KEY`, defaulting each to empty so a pre-set key is preserved.

**In one line:** declare the provider key slots from the environment, never hardcoded.

**What you'll see:** (no output - environment prep; keys are read silently and never printed)

Across seven runnable, keyless blocks you built the full reliability toolkit: the availability math that motivates everything (series multiplies down, parallel buys a nine back), a per-call timeout plus a deadline budget, transient-only retry with exponential backoff and full jitter, a circuit-breaker state machine, a health-tracked fallback chain with a degradation ladder, bulkheads and idempotency keys, and finally one composed resilient client where an open breaker fast-fails before any retry is wasted. Each block is deterministic logic, not an API call, because resilience is engineering, not luck. Next: Lesson 12.3 turns these same patterns into AI-gateway configuration, 12.4 makes caching the top rung of the degradation ladder, and 12.8 monitors the error budget you computed in block 1.